# Dissertation Data Collection Pipeline
## *From Tweets to Trades: Analysing Sentiment Signal and Information Network for Crypto Market Prediction*

---

This notebook collects **all data** required for the dissertation:

| # | Data Stream | Source | Credentials? |
|---|-----------|--------|-------------|
| 1 | Market OHLCV, price, volume | CryptoCompare API | No |
| 2 | Reddit posts & comments | Arctic Shift API | No |
| 3 | Twitter/X tweets | Published datasets (Kaggle) | Optional |

**Assets under study:**
- Bitcoin (BTC) and Ethereum (ETH) — Established cryptocurrencies
- Dogecoin (DOGE) and Shiba Inu (SHIB) — Speculative meme coins
- Tether (USDT) — Stablecoin control asset

**Time frame:** 1 January 2020 – 31 December 2024

**Instructions:** Run each section in order from top to bottom. At the end, download everything as a zip.

---

## 0. Setup & Dependencies

In [ ]:
!pip install -q requests pandas numpy

import requests
import pandas as pd
import numpy as np
import time
import os
import json
import re
import glob
from datetime import datetime, timezone, timedelta
from google.colab import files

print(f"pandas {pd.__version__}, numpy {np.__version__}")
print("All dependencies loaded.")

pandas 2.2.2, numpy 2.0.2
All dependencies loaded.


In [ ]:
# Create folder structure.
for d in ["data/raw/market", "data/raw/sentiment/reddit",
          "data/raw/sentiment/twitter/published", "data/raw/sentiment/news",
          "data/processed", "data/merged", "docs"]:
    os.makedirs(d, exist_ok=True)
print("Folder structure ready.")

Folder structure ready.


In [ ]:
# =============================================================
# SHARED CONFIGURATION
# =============================================================

START_DATE = datetime(2020, 1, 1, tzinfo=timezone.utc)
END_DATE   = datetime(2024, 12, 31, 23, 59, 59, tzinfo=timezone.utc)
START_TS   = int(START_DATE.timestamp())
END_TS     = int(END_DATE.timestamp())

ASSETS = {
    "bitcoin":   {"symbol": "BTC",  "category": "established"},
    "ethereum":  {"symbol": "ETH",  "category": "established"},
    "dogecoin":  {"symbol": "DOGE", "category": "meme_coin"},
    "shiba_inu": {"symbol": "SHIB", "category": "meme_coin"},
    "tether":    {"symbol": "USDT", "category": "stablecoin_control"},
}

ASSET_KEYWORDS = {
    "bitcoin":   ["bitcoin", "btc", "$btc", "#bitcoin"],
    "ethereum":  ["ethereum", "ether", "eth", "$eth", "#ethereum"],
    "dogecoin":  ["dogecoin", "doge", "$doge", "#dogecoin"],
    "shiba_inu": ["shiba", "shib", "$shib", "#shibarmy", "shiba inu", "shibainu"],
    "tether":    ["tether", "usdt", "$usdt"],
}

def detect_asset_mentions(text):
    if not text or not isinstance(text, str): return []
    text_lower = text.lower()
    mentioned = []
    for asset, keywords in ASSET_KEYWORDS.items():
        for kw in keywords:
            if kw.startswith(("$", "#")):
                if kw in text_lower: mentioned.append(asset); break
            elif len(kw) <= 4:
                if re.search(r'\b' + re.escape(kw) + r'\b', text_lower): mentioned.append(asset); break
            else:
                if kw in text_lower: mentioned.append(asset); break
    return mentioned

def clean_text(text):
    if not text or not isinstance(text, str): return ""
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\[deleted\]|\[removed\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(f"Study period: {START_DATE.date()} to {END_DATE.date()}")
print(f"Assets: {', '.join(f'{v["symbol"]}' for v in ASSETS.values())}")

Study period: 2020-01-01 to 2024-12-31
Assets: BTC, ETH, DOGE, SHIB, USDT


---
## 1. Market Data: CryptoCompare (Daily OHLCV, Price, Volume)

This section fetches daily Open-High-Low-Close-Volume (OHLCV) market data from the **CryptoCompare** API (https://min-api.cryptocompare.com). CryptoCompare was selected as the primary market data source because it provides full unrestricted historical data going back to each coin's launch date, requires no API key for basic daily endpoints, and has no regional access restrictions.

**Data collected per asset per day:**
- Open, High, Low, Close prices (USD)
- Trading volume (in both USD and base asset terms)
- Derived features: daily returns, log returns, 7-day and 30-day rolling volatility, intraday range

**No API key or credentials required.**

In [ ]:
# --- CryptoCompare functions ---

def fetch_cc_daily(fsym, tsym="USD", limit=2000, toTs=None):
    """Fetch daily OHLCV from CryptoCompare. Returns up to 2000 days per request."""
    url = "https://min-api.cryptocompare.com/data/v2/histoday"
    params = {"fsym": fsym, "tsym": tsym, "limit": limit}
    if toTs: params["toTs"] = toTs
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if data.get("Response") == "Error":
        raise Exception(data.get("Message", "Unknown CryptoCompare error"))
    return data["Data"]["Data"]

def collect_full_history(fsym, start_date, end_date, tsym="USD"):
    """Collect full daily history by paginating backwards from end_date."""
    all_records = []
    current_end_ts = int(end_date.timestamp())
    start_ts = int(start_date.timestamp())
    while True:
        records = fetch_cc_daily(fsym, tsym, limit=2000, toTs=current_end_ts)
        if not records: break
        filtered = [r for r in records if start_ts <= r["time"] <= int(end_date.timestamp())]
        all_records.extend(filtered)
        earliest = min(r["time"] for r in records)
        if earliest <= start_ts: break
        current_end_ts = earliest - 1
        time.sleep(1)
    # Deduplicate and sort.
    seen = set()
    unique = []
    for r in all_records:
        if r["time"] not in seen: seen.add(r["time"]); unique.append(r)
    unique.sort(key=lambda x: x["time"])
    return unique

def parse_cc(records, asset_name, symbol, category):
    """Parse CryptoCompare records into a clean DataFrame with derived features."""
    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["time"], unit="s")
    df["asset"], df["symbol"], df["category"] = asset_name, symbol, category
    df["price_usd"]        = df["close"]
    df["volume_usd"]       = df["volumeto"]
    df["daily_range_pct"]  = ((df["high"] - df["low"]) / df["open"].replace(0, np.nan)) * 100
    df["daily_return_pct"] = df["close"].pct_change() * 100
    df["log_return"]       = np.log(df["close"] / df["close"].shift(1))
    df["volatility_7d"]    = df["log_return"].rolling(7).std()
    df["volatility_30d"]   = df["log_return"].rolling(30).std()
    df["volume_change_pct"]= df["volumeto"].pct_change() * 100
    df = df[df["volumeto"] > 0]  # Remove days before coin had trading activity.
    out_cols = ["date","asset","symbol","category","open","high","low","close",
               "price_usd","volume_usd","volumefrom","daily_range_pct",
               "daily_return_pct","log_return","volatility_7d",
               "volatility_30d","volume_change_pct"]
    return df[out_cols].sort_values("date").reset_index(drop=True)

print("CryptoCompare functions loaded.")

CryptoCompare functions loaded.


In [ ]:
# --- Run market data collection ---
print("=" * 60)
print("MARKET DATA COLLECTION (CryptoCompare)")
print("=" * 60)

all_market = []
for name, info in ASSETS.items():
    print(f"\n>>> {name} ({info['symbol']})...")
    try:
        records = collect_full_history(info["symbol"], START_DATE, END_DATE)
        df = parse_cc(records, name, info["symbol"], info["category"])
        path = f"data/raw/market/{name}_daily.csv"
        df.to_csv(path, index=False)
        all_market.append(df)
        print(f"    {len(df)} days  |  {df['date'].min().date()} to {df['date'].max().date()}")
    except Exception as e:
        print(f"    ERROR: {e}")
    time.sleep(2)

if all_market:
    master_mkt = pd.concat(all_market, ignore_index=True)
    master_mkt.to_csv("data/raw/market/all_assets_daily.csv", index=False)
    print(f"\nMaster: data/raw/market/all_assets_daily.csv ({len(master_mkt)} rows)")
    display(master_mkt.groupby("asset").agg(
        days=("date","count"), start=("date","min"), end=("date","max"),
        mean_price=("price_usd","mean"), mean_volume=("volume_usd","mean")
    ).round(2))
    print("\nMarket data collection COMPLETE.")

MARKET DATA COLLECTION (CryptoCompare)

>>> bitcoin (BTC)...
    1827 days  |  2020-01-01 to 2024-12-31

>>> ethereum (ETH)...
    1827 days  |  2020-01-01 to 2024-12-31

>>> dogecoin (DOGE)...
    1827 days  |  2020-01-01 to 2024-12-31

>>> shiba_inu (SHIB)...
    1289 days  |  2021-06-22 to 2024-12-31

>>> tether (USDT)...
    1827 days  |  2020-01-01 to 2024-12-31

Master: data/raw/market/all_assets_daily.csv (8597 rows)


,days,start,end,mean_price,mean_volume
asset,,,,,
bitcoin,1827,2020-01-01,2024-12-31,36314.44,1.422144e+09
dogecoin,1827,2020-01-01,2024-12-31,0.11,7.740770e+07
ethereum,1827,2020-01-01,2024-12-31,1982.30,9.163719e+08
shiba_inu,1289,2021-06-22,2024-12-31,0.00,5.211047e+07
tether,1827,2020-01-01,2024-12-31,1.00,2.126922e+07



Market data collection COMPLETE.


---
## 2. Reddit Sentiment Data (Arctic Shift API)

Collects historical Reddit posts from crypto subreddits using the **Arctic Shift** API, the successor to the Pushshift archive.

**Target subreddits:**
- Asset-specific: `r/Bitcoin`, `r/ethereum`, `r/dogecoin`, `r/SHIBArmy`
- General: `r/CryptoCurrency`, `r/CryptoMarkets`, `r/SatoshiStreetBets`

**No credentials required.**

> Adjust `MAX_PER_SUB` below to control how many posts to collect per subreddit. Lower values run faster for testing.

In [ ]:
# --- Arctic Shift collection with EVEN temporal distribution ---

ARCTIC_SHIFT_BASE = "https://arctic-shift.photon-reddit.com"

def collect_arctic_shift_window(subreddit, start_epoch, end_epoch, max_records=1000):
    """Collect posts from Arctic Shift for a single time window."""
    url = f"{ARCTIC_SHIFT_BASE}/api/posts/search"
    records = []
    current_after = start_epoch

    while current_after < end_epoch and len(records) < max_records:
        params = {
            "subreddit": subreddit,
            "after": current_after,
            "before": end_epoch,
            "limit": 100,
            "sort": "asc",
            "sort_type": "created_utc"
        }
        try:
            resp = requests.get(url, params=params, timeout=30)
            if resp.status_code == 429:
                print("    Rate limited. Waiting 30s...")
                time.sleep(30); continue
            resp.raise_for_status()
            data = resp.json()
            results = data.get("data", [])
            if not results: break

            for item in results:
                created = item.get("created_utc", 0)
                full_text = f"{item.get('title', '')} {item.get('selftext', '')}"
                records.append({
                    "id": item.get("id", ""),
                    "created_utc": datetime.fromtimestamp(created, tz=timezone.utc).isoformat(),
                    "date": datetime.fromtimestamp(created, tz=timezone.utc).strftime("%Y-%m-%d"),
                    "subreddit": subreddit,
                    "title": item.get("title", ""),
                    "text": clean_text(item.get("selftext", "")),
                    "score": item.get("score", 0),
                    "num_comments": item.get("num_comments", 0),
                    "asset_mentions": json.dumps(detect_asset_mentions(full_text)),
                })

            current_after = results[-1].get("created_utc", current_after) + 1
            if len(results) < 100: break
            time.sleep(2)

        except requests.exceptions.RequestException as e:
            print(f"    Error: {e}")
            break

    return records


def collect_evenly_distributed(subreddit, start_date, end_date, total_budget=50000):
    """
    Collect posts evenly distributed across the study period.

    Divides the full time range into monthly windows and allocates
    an equal share of the total budget to each month. This ensures
    that every part of the study period is represented, rather than
    exhausting the entire budget on the earliest (or most active) months.
    """
    # Build monthly windows.
    windows = []
    current = start_date
    while current < end_date:
        # Calculate the start of the next month.
        if current.month == 12:
            next_month = current.replace(year=current.year + 1, month=1, day=1)
        else:
            next_month = current.replace(month=current.month + 1, day=1)
        window_end = min(next_month, end_date)
        windows.append((current, window_end))
        current = next_month

    num_windows = len(windows)
    per_window = max(total_budget // num_windows, 100)
    print(f"    Strategy: {num_windows} monthly windows, ~{per_window} posts each")

    all_records = []
    for i, (w_start, w_end) in enumerate(windows):
        w_start_ts = int(w_start.timestamp())
        w_end_ts = int(w_end.timestamp())
        month_label = w_start.strftime("%Y-%m")

        records = collect_arctic_shift_window(subreddit, w_start_ts, w_end_ts, max_records=per_window)
        all_records.extend(records)

        if (i + 1) % 6 == 0 or (i + 1) == num_windows:
            print(f"    Progress: {month_label} | {len(records)} posts this month | {len(all_records):,} total")

    return pd.DataFrame(all_records)

print(f"Arctic Shift functions loaded (even temporal distribution).")
print(f"Base URL: {ARCTIC_SHIFT_BASE}")

In [ ]:
# --- Configure and run Reddit collection ---

REDDIT_TARGETS = {
    "general":   ["CryptoCurrency", "CryptoMarkets", "SatoshiStreetBets"],
}

TOTAL_BUDGET_PER_SUB = 50000  # Total posts per subreddit, evenly spread across months.

print("=" * 60)
print("REDDIT DATA COLLECTION (Arctic Shift, evenly distributed)")
print("=" * 60)
print(f"Budget: {TOTAL_BUDGET_PER_SUB:,} posts per subreddit")
print(f"Period: {START_DATE.date()} to {END_DATE.date()}")

reddit_frames = []
for group, subs in REDDIT_TARGETS.items():
    for sub in subs:
        print(f"\n>>> r/{sub} (group: {group})...")
        df = collect_evenly_distributed(sub, START_DATE, END_DATE, total_budget=TOTAL_BUDGET_PER_SUB)
        if not df.empty:
            path = f"data/raw/sentiment/reddit/{group}_r{sub}_posts.csv"
            df.to_csv(path, index=False)
            reddit_frames.append(df)
            print(f"    DONE: {len(df):,} posts | {df['date'].min()} to {df['date'].max()}")
        else:
            print(f"    No data returned.")

if reddit_frames:
    master_reddit = pd.concat(reddit_frames, ignore_index=True)
    master_reddit = master_reddit.drop_duplicates(subset=["id"], keep="first")
    master_reddit.to_csv("data/raw/sentiment/reddit/all_reddit_posts.csv", index=False)
    print(f"\n{'=' * 60}")
    print(f"Master: data/raw/sentiment/reddit/all_reddit_posts.csv")
    print(f"Total unique posts: {len(master_reddit):,}")
    print(f"\nPer-subreddit breakdown:")
    for sub, cnt in master_reddit["subreddit"].value_counts().items():
        print(f"    r/{sub:25s} {cnt:>8,} posts")

    # Show temporal distribution to confirm even coverage.
    master_reddit["year_month"] = pd.to_datetime(master_reddit["date"]).dt.to_period("M")
    monthly = master_reddit.groupby(["subreddit", "year_month"]).size().unstack(level=0, fill_value=0)
    print(f"\nMonthly post counts (first and last 3 months):")
    display(monthly.head(3))
    print("...")
    display(monthly.tail(3))
    print("\nReddit collection COMPLETE.")

REDDIT DATA COLLECTION (Arctic Shift, evenly distributed)
Budget: 50,000 posts per subreddit
Period: 2020-01-01 to 2024-12-31

>>> r/Bitcoin (group: bitcoin)...
    Strategy: 60 monthly windows, ~833 posts each
    Progress: 2020-06 | 900 posts this month | 5,400 total
    Progress: 2020-12 | 900 posts this month | 10,800 total
    Progress: 2021-06 | 900 posts this month | 16,200 total
    Progress: 2021-12 | 900 posts this month | 21,600 total
    Progress: 2022-06 | 900 posts this month | 27,000 total
    Progress: 2022-12 | 900 posts this month | 32,400 total
    Progress: 2023-06 | 900 posts this month | 37,800 total
    Progress: 2023-12 | 900 posts this month | 43,200 total
    Progress: 2024-06 | 900 posts this month | 48,600 total
    Progress: 2024-12 | 900 posts this month | 54,000 total
    DONE: 54,000 posts | 2020-01-01 to 2024-12-03

>>> r/ethereum (group: ethereum)...
    Strategy: 60 monthly windows, ~833 posts each
    Progress: 2020-06 | 900 posts this month | 5,400 

subreddit,Bitcoin,CryptoCurrency,CryptoMarkets,SHIBArmy,SatoshiStreetBets,dogecoin,ethereum
year_month,,,,,,,
2020-01,900,900,900,0,0,454,900
2020-02,900,900,900,0,100,477,900
2020-03,900,900,900,0,108,547,900


...


subreddit,Bitcoin,CryptoCurrency,CryptoMarkets,SHIBArmy,SatoshiStreetBets,dogecoin,ethereum
year_month,,,,,,,
2024-10,900,900,900,559,476,900,900
2024-11,900,900,900,900,437,900,900
2024-12,900,900,900,900,459,900,900



Reddit collection COMPLETE.


---
## 3. Bluesky Sentiment Data (Public AT Protocol API)

Collects posts from Bluesky related to the five crypto assets using the public AT Protocol API. No API key, authentication, or account is required.

**Data coverage:** Bluesky launched its public beta in February 2023, so meaningful crypto discourse data is available from approximately mid-2023 onwards. This is positioned as a supplementary sentiment source enriching the Twitter and Reddit data for the latter portion of the study period.

**No credentials required.**

In [ ]:
# --- Bluesky collection functions (with authentication) ---

BSKY_API_BASE = "https://bsky.social/xrpc"
BSKY_PUBLIC_BASE = "https://public.api.bsky.app/xrpc"

BSKY_START = datetime(2023, 3, 1, tzinfo=timezone.utc)
BSKY_END   = datetime(2024, 12, 31, 23, 59, 59, tzinfo=timezone.utc)

BSKY_QUERIES = {
    "bitcoin":        ["bitcoin", "btc crypto", "bitcoin price"],
    "ethereum":       ["ethereum", "eth crypto", "ethereum price"],
    "dogecoin":       ["dogecoin", "doge crypto", "dogecoin price"],
    "shiba_inu":      ["shiba inu", "shib crypto", "shibarmy"],
    "crypto_general": ["cryptocurrency", "crypto market", "crypto regulation"],
}

MAX_POSTS_PER_QUERY_PER_MONTH = 2000

# ---------------------------------------------------------------
# ENTER YOUR BLUESKY CREDENTIALS HERE
# Create an app password at: Settings > App Passwords on bsky.app
# ---------------------------------------------------------------
BSKY_HANDLE = "ishaan28.bsky.social"       # <-- Your Bluesky handle
BSKY_APP_PASSWORD = "bacs-ln7p-jcgn-4m2t"     # <-- Your app password

def bsky_login():
    """Authenticate with Bluesky and return the access token."""
    resp = requests.post(
        f"{BSKY_API_BASE}/com.atproto.server.createSession",
        json={"identifier": BSKY_HANDLE, "password": BSKY_APP_PASSWORD},
    )
    resp.raise_for_status()
    session = resp.json()
    print(f"Logged in as: {session['handle']}")
    return session["accessJwt"]

def search_bsky(query, token, since=None, until=None, cursor=None):
    """Search Bluesky posts using authenticated request."""
    params = {"q": query, "limit": 25, "sort": "latest"}
    if since: params["since"] = since
    if until: params["until"] = until
    if cursor: params["cursor"] = cursor
    headers = {"Authorization": f"Bearer {token}"}
    try:
        resp = requests.get(
            f"{BSKY_API_BASE}/app.bsky.feed.searchPosts",
            params=params, headers=headers, timeout=30
        )
        if resp.status_code == 429:
            print("    Rate limited. Waiting 60s...")
            time.sleep(60)
            return search_bsky(query, token, since, until, cursor)
        if resp.status_code == 401:
            print("    Token expired. Re-authenticating...")
            new_token = bsky_login()
            return search_bsky(query, new_token, since, until, cursor)
        resp.raise_for_status()
        data = resp.json()
        return data.get("posts", []), data.get("cursor", None), token
    except requests.exceptions.RequestException as e:
        print(f"    API error: {e}")
        return [], None, token

def parse_bsky_post(post):
    """Parse a raw Bluesky post into a clean dictionary."""
    record = post.get("record", {})
    author = post.get("author", {})
    text = record.get("text", "")
    created_at = record.get("createdAt", "")
    try:
        dt = datetime.fromisoformat(created_at.replace("Z", "+00:00"))
        date_str = dt.strftime("%Y-%m-%d")
    except (ValueError, AttributeError):
        date_str = ""
    langs = record.get("langs", [])
    return {
        "id": post.get("uri", ""),
        "created_at": created_at,
        "date": date_str,
        "text": clean_text(text),
        "author_handle": author.get("handle", ""),
        "like_count": post.get("likeCount", 0),
        "repost_count": post.get("repostCount", 0),
        "reply_count": post.get("replyCount", 0),
        "language": langs[0] if langs else "",
        "asset_mentions": json.dumps(detect_asset_mentions(text)),
    }

def collect_bsky_window(query, token, w_start, w_end, max_posts=200):
    """Collect posts for one query within one monthly time window."""
    since_str = w_start.strftime("%Y-%m-%dT%H:%M:%SZ")
    until_str = w_end.strftime("%Y-%m-%dT%H:%M:%SZ")
    all_posts = []
    cursor = None
    current_token = token
    while len(all_posts) < max_posts:
        raw_posts, next_cursor, current_token = search_bsky(
            query, current_token, since_str, until_str, cursor
        )
        if not raw_posts: break
        for p in raw_posts:
            parsed = parse_bsky_post(p)
            if parsed["language"] in ("", "en", "en-US", "en-GB"):
                all_posts.append(parsed)
        cursor = next_cursor
        if not cursor: break
        time.sleep(2)
    return all_posts[:max_posts], current_token

def bsky_monthly_windows(start_date, end_date):
    """Generate monthly (start, end) window tuples."""
    windows = []
    current = start_date
    while current < end_date:
        if current.month == 12:
            next_m = current.replace(year=current.year + 1, month=1, day=1)
        else:
            next_m = current.replace(month=current.month + 1, day=1)
        window_end = min(next_m - timedelta(seconds=1), end_date)
        windows.append((current, window_end))
        current = next_m
    return windows

# Authenticate now.
if BSKY_HANDLE == "your-handle.bsky.social":
    print("*** Please enter your Bluesky credentials above ***")
    print("Create a free account at bsky.app if you don't have one.")
    print("Then go to Settings > App Passwords to create an app password.")
    bsky_token = None
else:
    bsky_token = bsky_login()
    print(f"Period: {BSKY_START.date()} to {BSKY_END.date()}")

Logged in as: ishaan28.bsky.social
Period: 2023-03-01 to 2024-12-31


In [ ]:
# --- Run Bluesky collection ---
if bsky_token is None:
    print("Bluesky credentials not configured. Skipping collection.")
else:
    print("=" * 60)
    print("BLUESKY SENTIMENT DATA COLLECTION")
    print("=" * 60)

    os.makedirs("data/raw/sentiment/bluesky", exist_ok=True)
    windows = bsky_monthly_windows(BSKY_START, BSKY_END)
    print(f"Monthly windows: {len(windows)}")
    print(f"Budget: {MAX_POSTS_PER_QUERY_PER_MONTH} posts/query/month\n")

    bsky_frames = []
    current_token = bsky_token

    for category, queries in BSKY_QUERIES.items():
        print(f"\n>>> Category: {category}")
        category_posts = []

        for query in queries:
            print(f"  Query: \"{query}\"")
            for w_start, w_end in windows:
                month_label = w_start.strftime("%Y-%m")
                posts, current_token = collect_bsky_window(
                    query, current_token, w_start, w_end,
                    max_posts=MAX_POSTS_PER_QUERY_PER_MONTH
                )
                for p in posts:
                    p["search_query"] = query
                    p["category"] = category
                    p["month"] = month_label
                category_posts.extend(posts)
                if posts:
                    print(f"    {month_label}: {len(posts)} posts (total: {len(category_posts):,})")
                time.sleep(3)

        if category_posts:
            df = pd.DataFrame(category_posts)
            df = df.drop_duplicates(subset=["id"], keep="first")
            path = f"data/raw/sentiment/bluesky/{category}_bluesky.csv"
            df.to_csv(path, index=False)
            bsky_frames.append(df)
            print(f"  Saved: {path} ({len(df):,} unique posts)")

    if bsky_frames:
        master_bsky = pd.concat(bsky_frames, ignore_index=True)
        master_bsky = master_bsky.drop_duplicates(subset=["id"], keep="first")
        master_bsky.to_csv("data/raw/sentiment/bluesky/all_bluesky_posts.csv", index=False)
        print(f"\n{'=' * 60}")
        print(f"Master: data/raw/sentiment/bluesky/all_bluesky_posts.csv")
        print(f"Total unique posts: {len(master_bsky):,}")
        print(f"Date range: {master_bsky['date'].min()} to {master_bsky['date'].max()}")
        print(f"\nPer-category:")
        for cat, cnt in master_bsky["category"].value_counts().items():
            print(f"  {cat:20s} {cnt:>8,} posts")
        print("\nBluesky collection COMPLETE.")
    else:
        print("No Bluesky data collected.")

BLUESKY SENTIMENT DATA COLLECTION
Monthly windows: 22
Budget: 2000 posts/query/month


>>> Category: bitcoin
  Query: "bitcoin"
    2023-03: 61 posts (total: 61)
    2023-04: 2000 posts (total: 2,061)
    2023-05: 1086 posts (total: 3,147)
    2023-06: 669 posts (total: 3,816)
    2023-07: 1476 posts (total: 5,292)
    2023-08: 1027 posts (total: 6,319)
    2023-09: 1458 posts (total: 7,777)
    2023-10: 2000 posts (total: 9,777)
    2023-11: 2000 posts (total: 11,777)
    2023-12: 2000 posts (total: 13,777)
    2024-01: 2000 posts (total: 15,777)
    2024-02: 2000 posts (total: 17,777)
    2024-03: 2000 posts (total: 19,777)
    2024-04: 2000 posts (total: 21,777)
    2024-05: 2000 posts (total: 23,777)
    2024-06: 2000 posts (total: 25,777)
    2024-07: 2000 posts (total: 27,777)
    2024-08: 2000 posts (total: 29,777)
    2024-09: 2000 posts (total: 31,777)
    2024-10: 2000 posts (total: 33,777)
    2024-11: 2000 posts (total: 35,777)
    2024-12: 2000 posts (total: 37,777)
  Quer

KeyboardInterrupt: 

---
## 4. Download All Collected Data

Run the cell below to zip everything and download it to your computer.

In [ ]:
import shutil

zip_name = "dissertation_collected_data"
shutil.make_archive(zip_name, 'zip', '.', 'data')

zip_path = f"{zip_name}.zip"
size_mb = os.path.getsize(zip_path) / 1e6
print(f"Archive: {zip_path} ({size_mb:.1f} MB)")
print("Downloading...")
files.download(zip_path)
print("Done! Extract into your dissertation_project/ directory.")

---
## Next Steps

After downloading the collected data:

1. **Data preprocessing** — Clean, align timestamps, handle missing values
2. **Sentiment analysis** — Run the LLM-based classifier (Ollama) on the text data
3. **Merge datasets** — Create a unified daily time-series combining market and sentiment data
4. **Analysis** — VAR, Granger causality, transfer entropy, and predictive modelling

---
*Notebook created for BASC0024 Final Year Dissertation, UCL Arts and Sciences.*